[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/08_Model_Optimization_and_Quantization/01_Graph_Optimizations/Graph_Optimizations_Deep_Dive.ipynb)

# 8.1 Graph Optimizations — Deep Dive

![Graph Optimization Techniques](assets/graph_optimization_techniques.png)

## Table of Contents
1. [Why Optimize Computation Graphs](#section-1)
2. [Constant Folding (Partial Evaluation)](#section-2)
3. [Dead Code Elimination](#section-3)
4. [Operator Fusion — Theory](#section-4)
5. [Conv + BatchNormalization Fusion (Full Derivation)](#section-5)
6. [MatMul + Add + Relu Fusion with Memory Traffic Analysis](#section-6)
7. [Pattern Matching and Graph Rewrite Rules](#section-7)
8. [ORT Optimization Levels](#section-8)
9. [Correctness Proofs and Semantic Equivalence](#section-9)
10. [Practical Implementation](#section-10)
11. [Summary](#section-11)

<a id='section-1'></a>
## Section 1: Why Optimize Computation Graphs

### The Optimization Imperative

When a model is exported to ONNX, the resulting graph faithfully represents **training-time** computation. Training graphs are verbose:

- Dropout nodes (no-ops at inference)
- Separate BatchNorm layers that could be folded into convolutions
- Constant subexpressions evaluated at every forward pass
- Dead branches from conditional logic

### Cost Model

For a graph with $n$ nodes, each node incurs:

$$\text{Cost}(\text{node}_i) = \underbrace{\tau_{\text{launch}}}_{\text{kernel launch}} + \underbrace{\tau_{\text{compute}}}_{\text{arithmetic}} + \underbrace{\tau_{\text{mem}}}_{\text{memory I/O}}$$

For memory-bound ops (most element-wise), $\tau_{\text{mem}} \gg \tau_{\text{compute}}$. Each intermediate tensor requires:
- **Write** from producer: $|T| \times \text{sizeof}(\text{dtype})$ bytes
- **Read** by consumer: same cost

### Fusion Memory Savings

When $k$ consecutive memory-bound ops are fused into one kernel:

$$\text{Savings} = (k-1) \times 2 \times |\text{tensor}| \times \text{sizeof}(\text{dtype})$$

The factor of 2 accounts for the eliminated write + read of each intermediate tensor.

### Optimization Pipeline Overview

```
┌─────────────────────────────────────────────────────────────────────┐
│                    Graph Optimization Pipeline                       │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  Raw ONNX Graph                                                     │
│       │                                                             │
│       ▼                                                             │
│  ┌──────────────────┐                                               │
│  │ Constant Folding  │ ── Evaluate static subgraphs                 │
│  └────────┬─────────┘                                               │
│           ▼                                                         │
│  ┌──────────────────┐                                               │
│  │ Dead Code Elim.   │ ── Remove unreachable nodes                  │
│  └────────┬─────────┘                                               │
│           ▼                                                         │
│  ┌──────────────────┐                                               │
│  │ Identity Removal  │ ── Bypass no-op nodes                        │
│  └────────┬─────────┘                                               │
│           ▼                                                         │
│  ┌──────────────────┐                                               │
│  │ Operator Fusion   │ ── Conv+BN, Conv+Relu, MatMul+Add, etc.     │
│  └────────┬─────────┘                                               │
│           ▼                                                         │
│  Optimized ONNX Graph                                               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
!pip install onnx onnxruntime onnxoptimizer numpy matplotlib -q

In [ ]:
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model
import matplotlib.pyplot as plt
import time

print(f"ONNX version: {onnx.__version__}")

try:
    import onnxoptimizer
    print(f"onnxoptimizer version: {onnxoptimizer.__version__}")
except ImportError:
    print("onnxoptimizer not available")

try:
    import onnxruntime as ort
    print(f"onnxruntime version: {ort.__version__}")
except ImportError:
    print("onnxruntime not available")

<a id='section-2'></a>
## Section 2: Constant Folding (Partial Evaluation)

### Formal Definition

Constant folding is a special case of **partial evaluation**. Given a computation graph $G$ and a known-value environment $\sigma$ mapping names to constant tensors:

$$\text{PE}(G, \sigma) \to G'$$

where $G'$ is a **residual graph** — the portion of $G$ that depends on runtime inputs.

**Definition.** A node $v$ in $G$ is **foldable** under $\sigma$ if:

$$\forall i \in \text{inputs}(v): i \in \text{dom}(\sigma) \lor i \in \text{outputs}(v') \text{ for some foldable } v'$$

That is, all inputs are either directly known constants or produced by other foldable nodes.

### Correctness Proof

**Theorem.** If $\text{PE}(G, \sigma) = G'$, then $\forall x \in \mathcal{X}: G(x) = G'(x)$.

**Proof.** Let $v$ be a foldable node with $\text{op}(v) = f$ and $\text{inputs}(v) = (c_1, \ldots, c_k)$ where each $c_i \in \sigma$.

1. By ONNX specification, $f$ is **deterministic** for standard ops.
2. Since all $c_i$ are immutable initializers, $f(c_1, \ldots, c_k) = c_{\text{result}}$ is a fixed value.
3. Replacing node $v$ with constant $c_{\text{result}}$ in $\sigma$ preserves semantics because any consumer of $v$'s output receives the same value $c_{\text{result}}$.
4. By induction over topological order, all foldable nodes can be replaced. $\square$

### Cascading Folds (Fixed-Point Iteration)

Folding is applied iteratively until a fixed point:

$$G_0 \xrightarrow{\text{fold}} G_1 \xrightarrow{\text{fold}} \cdots \xrightarrow{\text{fold}} G_n = G_{n+1}$$

Each fold may expose new foldable subgraphs. For example, folding `Add(c1, c2) → c3` makes `Mul(c3, c4)` foldable.

**Termination.** Each iteration strictly reduces the number of non-constant nodes (or terminates). Since node count is bounded, the algorithm terminates in $\leq |V|$ iterations.

### Before / After Diagram

```
BEFORE constant folding:              AFTER constant folding:

  c1 ──┐                               c_result (precomputed = c1+c2)
       ├──▶ Add ──▶ c12 ──┐                   │
  c2 ──┘                  │                   │
                          ├──▶ Mul ──▶ y      ├──▶ Mul ──▶ y
  x ─────────────────────┘                   │
                                        x ───┘

  Nodes: 2 (Add, Mul)                  Nodes: 1 (Mul)
  Runtime ops: 2                       Runtime ops: 1
  Constants: {c1, c2}                  Constants: {c_result}
```

### Cascading Example

```
ITERATION 0:                 ITERATION 1:               ITERATION 2 (fixed point):

  c1 ─┐                      c3=c1+c2 ─┐                c5=c3*c4 ─┐
      ├─▶ Add ─▶ c3          (folded)  ├─▶ Mul ─▶ c5    (folded)  ├─▶ Sub ─▶ y
  c2 ─┘       │                        │                          │
              ├─▶ Mul ─▶ c5    c4 ──────┘                 x ───────┘
  c4 ────────┘       │
                    ├─▶ Sub ─▶ y
  x ────────────────┘

  3 ops (Add, Mul, Sub)     2 ops (Mul, Sub)            1 op (Sub)
```

In [ ]:
# Build a model with cascading constant-foldable subgraphs
# Computation: y = x - ((a + b) * c) where a=2.0, b=3.0, c=4.0
# Fold 1: a+b = 5.0, Fold 2: 5.0*c = 20.0, Result: y = x - 20.0

np.random.seed(42)

a_val = np.array([2.0], dtype=np.float32)
b_val = np.array([3.0], dtype=np.float32)
c_val = np.array([4.0], dtype=np.float32)

a_init = numpy_helper.from_array(a_val, name='a')
b_init = numpy_helper.from_array(b_val, name='b')
c_init = numpy_helper.from_array(c_val, name='c')

add_node = helper.make_node('Add', ['a', 'b'], ['ab'])
mul_node = helper.make_node('Mul', ['ab', 'c'], ['abc'])
sub_node = helper.make_node('Sub', ['x', 'abc'], ['y'])

x_info = helper.make_tensor_value_info('x', TensorProto.FLOAT, [1])
y_info = helper.make_tensor_value_info('y', TensorProto.FLOAT, [1])

graph = helper.make_graph(
    [add_node, mul_node, sub_node], 'cascading_fold_demo',
    inputs=[x_info], outputs=[y_info],
    initializer=[a_init, b_init, c_init]
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
check_model(model)

print("BEFORE constant folding:")
print(f"  Nodes: {len(model.graph.node)}")
for n in model.graph.node:
    print(f"    {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")
print(f"  Initializers: {[i.name for i in model.graph.initializer]}")

# Apply constant folding via ORT
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
so.optimized_model_filepath = '/tmp/folded_cascading.onnx'
sess = ort.InferenceSession(model.SerializeToString(), so)
folded = onnx.load('/tmp/folded_cascading.onnx')

print(f"\nAFTER constant folding (ORT BASIC):")
print(f"  Nodes: {len(folded.graph.node)}")
for n in folded.graph.node:
    print(f"    {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")
print(f"  Initializers: {[i.name for i in folded.graph.initializer]}")
for init in folded.graph.initializer:
    val = numpy_helper.to_array(init)
    print(f"    {init.name} = {val}")

# Verify equivalence
x_test = np.array([10.0], dtype=np.float32)
out_orig = sess.run(None, {'x': x_test})[0]
expected = x_test - (a_val + b_val) * c_val
print(f"\nVerification: f(10.0) = 10.0 - (2+3)*4 = {expected[0]}")
print(f"Model output: {out_orig[0]}")
print(f"Match: {np.allclose(out_orig, expected)}")

<a id='section-3'></a>
## Section 3: Dead Code Elimination

### Formal Definition via Reachability

Dead code elimination (DCE) removes nodes whose outputs are **never consumed** by any graph output.

### Algorithm: Backward Reachability (Live Variable Analysis)

**Input:** Graph $G = (V, E)$ with output set $O \subseteq V$

**Algorithm:**
1. Initialize $\text{Live} = \emptyset$
2. For each output tensor $o \in \text{graph.output}$: add $o$ to worklist $W$
3. While $W \neq \emptyset$:
   - Pop tensor $t$ from $W$
   - Find producer node $v$ of $t$
   - If $v \notin \text{Live}$: add $v$ to $\text{Live}$, add all $\text{inputs}(v)$ to $W$
4. Remove all $v \in V \setminus \text{Live}$

**Formal Criterion.** A node $v$ is **dead** if:

$$\nexists \text{ directed path } v \to o \text{ for any output } o \in \text{graph.output}$$

Equivalently:

$$v \text{ is dead} \iff \text{outputs}(v) \cap \text{LiveTensors} = \emptyset$$

### Complexity

The backward reachability analysis runs in $O(|V| + |E|)$ — linear in graph size.

### Before / After

```
BEFORE dead code elimination:          AFTER dead code elimination:

  x ──▶ Relu ──▶ r ──▶ Sigmoid ──▶ y    x ──▶ Relu ──▶ r ──▶ Sigmoid ──▶ y
  │                                    
  ├──▶ Exp ──▶ e ──▶ Log ──▶ dead       (Exp, Log removed — outputs unused)
  │
  └──▶ Neg ──▶ n  (unused)               (Neg removed)

  Nodes: 5                               Nodes: 2
  Output: [y]                            Output: [y]
```

### Interaction with Other Passes

DCE often runs **after** other passes because:
- Constant folding can make consumer nodes dead (if their outputs were only used by folded expressions)
- Pattern replacement may leave orphaned nodes from the original pattern

In [ ]:
# Build a model with dead branches and demonstrate elimination
x_info = helper.make_tensor_value_info('x', TensorProto.FLOAT, [4])
y_info = helper.make_tensor_value_info('y', TensorProto.FLOAT, [4])

# Live path: x -> Relu -> Sigmoid -> y
relu_node = helper.make_node('Relu', ['x'], ['r'])
sigmoid_node = helper.make_node('Sigmoid', ['r'], ['y'])

# Dead path 1: x -> Exp -> Log -> dead_output (not in graph outputs)
exp_node = helper.make_node('Exp', ['x'], ['e'])
log_node = helper.make_node('Log', ['e'], ['dead_out1'])

# Dead path 2: x -> Neg -> Abs -> dead_output2
neg_node = helper.make_node('Neg', ['x'], ['neg_x'])
abs_node = helper.make_node('Abs', ['neg_x'], ['dead_out2'])

graph = helper.make_graph(
    [relu_node, sigmoid_node, exp_node, log_node, neg_node, abs_node],
    'dead_code_demo',
    inputs=[x_info], outputs=[y_info]
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])

print("BEFORE dead code elimination:")
print(f"  Total nodes: {len(model.graph.node)}")
for n in model.graph.node:
    print(f"    {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")

# Manual backward reachability analysis
print("\n--- Live Variable Analysis ---")
live_tensors = set(['y'])  # Start from graph outputs
live_nodes = []
all_nodes = list(model.graph.node)

changed = True
while changed:
    changed = False
    for node in all_nodes:
        if node in live_nodes:
            continue
        if any(out in live_tensors for out in node.output):
            live_nodes.append(node)
            for inp in node.input:
                live_tensors.add(inp)
            changed = True

dead_nodes = [n for n in all_nodes if n not in live_nodes]
print(f"  Live nodes: {[n.op_type for n in live_nodes]}")
print(f"  Dead nodes: {[n.op_type for n in dead_nodes]}")
print(f"  Live tensors: {live_tensors}")

# Apply DCE via onnxoptimizer
opt_model = onnxoptimizer.optimize(model, ['eliminate_deadend'])
print(f"\nAFTER dead code elimination (onnxoptimizer):")
print(f"  Total nodes: {len(opt_model.graph.node)}")
for n in opt_model.graph.node:
    print(f"    {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")
print(f"\n  Nodes removed: {len(model.graph.node) - len(opt_model.graph.node)}")

<a id='section-4'></a>
## Section 4: Operator Fusion — Theory

### Formal Definition

Operator fusion combines a sequence of nodes $(v_1, v_2, \ldots, v_k)$ into a single fused node $v_f$ such that:

$$\text{semantics}(v_f) = v_k \circ v_{k-1} \circ \cdots \circ v_1$$

### Necessary Conditions for Fusion

A sequence $(v_1, \ldots, v_k)$ is **fusible** if:
1. **Linear dependence**: $\text{output}(v_i) = \text{input}(v_{i+1})$ for $i < k$
2. **Single consumer**: $v_i$'s output has exactly one consumer (otherwise fusion changes computation order)
3. **Kernel exists**: The runtime provides a fused kernel for the pattern
4. **Shape compatibility**: All intermediate shapes satisfy the fused kernel's constraints

### Why Fusion Helps: The Roofline Perspective

For a sequence of memory-bound operators with arithmetic intensity $I_i = \frac{\text{FLOPs}_i}{\text{Bytes}_i}$:

**Unfused time** (each op is memory-bound):
$$T_{\text{unfused}} = \sum_{i=1}^k \frac{\text{Bytes}_i}{\text{BW}} = \frac{1}{\text{BW}} \sum_{i=1}^k (|\text{in}_i| + |\text{out}_i|) \times \text{sizeof}$$

**Fused time** (one kernel, intermediates stay in registers/L1):
$$T_{\text{fused}} = \frac{|\text{in}_1| + |\text{out}_k|}{\text{BW}} \times \text{sizeof}$$

**Speedup** from fusing $k$ element-wise ops on tensor of size $|T|$:

$$\text{Speedup} = \frac{T_{\text{unfused}}}{T_{\text{fused}}} = \frac{2k \cdot |T|}{2 \cdot |T|} = k$$

Fusing $k$ memory-bound ops gives up to $k\times$ speedup!

### Memory Traffic Analysis

```
UNFUSED: 3 element-wise ops (Relu → Add(bias) → Sigmoid)
on tensor of size |T| = N×C×H×W

  x ──[read |T|]──▶ Relu ──[write |T|]──▶ buffer₁
  buffer₁ ──[read |T|]──▶ Add ──[write |T|]──▶ buffer₂  
  buffer₂ ──[read |T|]──▶ Sigmoid ──[write |T|]──▶ y

  Total memory traffic: 6×|T|×sizeof (3 reads + 3 writes)

FUSED: single kernel

  x ──[read |T|]──▶ FusedKernel(relu→add→sigmoid) ──[write |T|]──▶ y

  Total memory traffic: 2×|T|×sizeof (1 read + 1 write)
  Savings: 4×|T|×sizeof = (k-1)×2×|T|×sizeof
```

<a id='section-5'></a>
## Section 5: Conv + BatchNormalization Fusion — Full Derivation

### The Most Important Fusion in Deep Learning Inference

BatchNormalization stabilizes training but at inference uses **fixed** running statistics. This means it can be algebraically absorbed into preceding Conv weights.

### Setup

**Conv layer** with weight $W \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}} \times k_h \times k_w}$ and bias $b \in \mathbb{R}^{C_{\text{out}}}$:

$$h_c = \text{Conv}(x)_c = \sum_{c'} W_{c,c'} * x_{c'} + b_c$$

**BatchNorm layer** with learned parameters $\gamma, \beta \in \mathbb{R}^{C_{\text{out}}}$ and running statistics $\mu, \sigma^2 \in \mathbb{R}^{C_{\text{out}}}$:

$$\text{BN}(h)_c = \gamma_c \frac{h_c - \mu_c}{\sqrt{\sigma^2_c + \epsilon}} + \beta_c$$

### Complete Derivation

**Step 1:** Compose BN after Conv:

$$y_c = \gamma_c \frac{\left(\sum_{c'} W_{c,c'} * x_{c'} + b_c\right) - \mu_c}{\sqrt{\sigma^2_c + \epsilon}} + \beta_c$$

**Step 2:** Define per-channel scale $s_c = \frac{\gamma_c}{\sqrt{\sigma^2_c + \epsilon}}$

**Step 3:** Expand and collect terms:

$$y_c = s_c \left(\sum_{c'} W_{c,c'} * x_{c'}\right) + s_c(b_c - \mu_c) + \beta_c$$

**Step 4:** Since convolution is linear, $s_c$ can be absorbed into weights:

$$y_c = \sum_{c'} (s_c \cdot W_{c,c'}) * x_{c'} + \underbrace{s_c(b_c - \mu_c) + \beta_c}_{b'_c}$$

### Fused Parameters

$$\boxed{W'_{c} = \frac{\gamma_c}{\sqrt{\sigma^2_c + \epsilon}} \cdot W_c}$$

$$\boxed{b'_c = \frac{\gamma_c (b_c - \mu_c)}{\sqrt{\sigma^2_c + \epsilon}} + \beta_c}$$

### Data Flow Diagram

```
BEFORE Conv+BN fusion:

  x ──▶ Conv(W, b) ──▶ h ──▶ BN(γ, β, μ, σ²) ──▶ y
              │                       │
              │ write h to DRAM       │ read h from DRAM
              │ (N×Cout×H×W×4 bytes)  │ + write y to DRAM
              │                       │
              └── kernel launch #1 ───┘── kernel launch #2

AFTER Conv+BN fusion:

  x ──▶ Conv(W', b') ──▶ y
              │
              │ single kernel, no intermediate buffer
              │ BN is FREE (absorbed into weights)
              │
              └── kernel launch #1 only

  W' = (γ / √(σ²+ε)) ⊙ W    (one-time precomputation)
  b' = (γ / √(σ²+ε)) ⊙ (b-μ) + β
```

### Savings Summary

| Metric | Before (Conv + BN) | After (Fused Conv) |
|--------|-------------------|--------------------|   
| Kernel launches | 2 | 1 |
| Memory traffic | $+2 \times N \times C_{\text{out}} \times H \times W \times 4$ bytes extra | 0 extra |
| Parameters stored | $W, b, \gamma, \beta, \mu, \sigma^2$ | $W', b'$ only |
| Latency | $\tau_{\text{conv}} + \tau_{\text{bn}}$ | $\tau_{\text{conv}}$ (BN is free) |

In [ ]:
# Implement Conv+BN fusion from scratch and verify numerical equivalence
np.random.seed(42)

C_out, C_in, kH, kW = 16, 3, 3, 3
W_conv = np.random.randn(C_out, C_in, kH, kW).astype(np.float32) * 0.1
b_conv = np.random.randn(C_out).astype(np.float32) * 0.01

gamma = np.random.rand(C_out).astype(np.float32) * 0.5 + 0.75
beta = np.random.randn(C_out).astype(np.float32) * 0.1
mean = np.random.randn(C_out).astype(np.float32) * 0.5
var = np.abs(np.random.randn(C_out).astype(np.float32)) + 0.1
epsilon = 1e-5

# Compute fused weights using derived formulas
scale = gamma / np.sqrt(var + epsilon)
W_fused = W_conv * scale.reshape(C_out, 1, 1, 1)
b_fused = scale * (b_conv - mean) + beta

print("Conv+BN Fusion Implementation")
print("=" * 50)
print(f"\nOriginal parameters:")
print(f"  W shape: {W_conv.shape} ({W_conv.size} params)")
print(f"  b shape: {b_conv.shape} ({b_conv.size} params)")
print(f"  BN: gamma={gamma.shape}, beta={beta.shape}, mu={mean.shape}, var={var.shape}")
print(f"  Total params stored: {W_conv.size + b_conv.size + 4*C_out}")
print(f"\nFused parameters:")
print(f"  W' shape: {W_fused.shape} ({W_fused.size} params)")
print(f"  b' shape: {b_fused.shape} ({b_fused.size} params)")
print(f"  Total params stored: {W_fused.size + b_fused.size}")
print(f"\nScale factors s_c (first 5): {scale[:5].round(4)}")
print(f"\nMemory savings per inference (for 32×32 spatial):")
H_out, W_out = 32, 32
intermediate_bytes = 1 * C_out * H_out * W_out * 4
print(f"  Eliminated intermediate buffer: {intermediate_bytes/1024:.1f} KB")
print(f"  Eliminated memory traffic: {2*intermediate_bytes/1024:.1f} KB (write+read)")

In [ ]:
# Verify numerical equivalence between Conv+BN and FusedConv

def simple_conv2d(x, w, b, padding=1):
    """Naive 2D convolution for verification."""
    N, C_in, H, W_in = x.shape
    C_out, _, kH, kW = w.shape
    H_out = H + 2*padding - kH + 1
    W_out = W_in + 2*padding - kW + 1
    x_pad = np.pad(x, ((0,0), (0,0), (padding,padding), (padding,padding)))
    out = np.zeros((N, C_out, H_out, W_out), dtype=np.float32)
    for co in range(C_out):
        for i in range(H_out):
            for j in range(W_out):
                patch = x_pad[:, :, i:i+kH, j:j+kW]
                out[:, co, i, j] = np.sum(patch * w[co], axis=(1,2,3)) + b[co]
    return out

def batch_norm_inference(x, gamma, beta, mean, var, eps=1e-5):
    """Apply batch normalization in inference mode."""
    s = gamma / np.sqrt(var + eps)
    return x * s.reshape(1, -1, 1, 1) + (beta - mean * s).reshape(1, -1, 1, 1)

# Use small spatial dims for numpy verification
x_test = np.random.randn(1, C_in, 6, 6).astype(np.float32)

# Path 1: Conv -> BN (original, two operations)
conv_out = simple_conv2d(x_test, W_conv, b_conv, padding=1)
original_out = batch_norm_inference(conv_out, gamma, beta, mean, var, epsilon)

# Path 2: Fused Conv (single operation with derived W', b')
fused_out = simple_conv2d(x_test, W_fused, b_fused, padding=1)

max_diff = np.max(np.abs(original_out - fused_out))
mean_diff = np.mean(np.abs(original_out - fused_out))
rel_diff = np.max(np.abs(original_out - fused_out) / (np.abs(original_out) + 1e-8))

print("Numerical Equivalence Verification: Conv+BN = FusedConv")
print("=" * 55)
print(f"\nOutput shape: {original_out.shape}")
print(f"Max absolute difference:  {max_diff:.2e}")
print(f"Mean absolute difference: {mean_diff:.2e}")
print(f"Max relative difference:  {rel_diff:.2e}")
print(f"\nWithin float32 tolerance (1e-5): {max_diff < 1e-5}")
print(f"\nConclusion: BN(Conv_{{W,b}}(x)) ≡ Conv_{{W',b'}}(x)  ✓")
print(f"  where W' = (γ/√(σ²+ε))·W, b' = (γ/√(σ²+ε))·(b-μ) + β")

In [ ]:
# Build ONNX Conv+BN model and apply fusion via ORT
C_in, C_out, kH, kW = 3, 32, 3, 3

W_init = numpy_helper.from_array(np.random.randn(C_out, C_in, kH, kW).astype(np.float32)*0.1, 'W')
b_init = numpy_helper.from_array(np.random.randn(C_out).astype(np.float32)*0.01, 'b')
bn_scale = numpy_helper.from_array(np.ones(C_out, dtype=np.float32)*0.9, 'scale')
bn_bias = numpy_helper.from_array(np.zeros(C_out, dtype=np.float32), 'bias')
bn_mean = numpy_helper.from_array(np.random.randn(C_out).astype(np.float32)*0.1, 'mean')
bn_var = numpy_helper.from_array(np.abs(np.random.randn(C_out).astype(np.float32))+0.5, 'var')

conv_node = helper.make_node('Conv', ['X', 'W', 'b'], ['h'], kernel_shape=[3,3], pads=[1,1,1,1])
bn_node = helper.make_node('BatchNormalization', ['h', 'scale', 'bias', 'mean', 'var'], ['Y'])

X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, C_in, 64, 64])
Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, C_out, 64, 64])

graph = helper.make_graph(
    [conv_node, bn_node], 'conv_bn_model',
    inputs=[X_info], outputs=[Y_info],
    initializer=[W_init, b_init, bn_scale, bn_bias, bn_mean, bn_var]
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
check_model(model)

print(f"Before fusion: {len(model.graph.node)} nodes")
for n in model.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)})")

# Apply extended optimization (where Conv+BN fusion lives)
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
so.optimized_model_filepath = '/tmp/conv_bn_fused.onnx'
sess = ort.InferenceSession(model.SerializeToString(), so)
fused_model = onnx.load('/tmp/conv_bn_fused.onnx')

print(f"\nAfter ORT EXTENDED fusion: {len(fused_model.graph.node)} nodes")
for n in fused_model.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)})")
print(f"\nBatchNormalization node eliminated: {'BatchNormalization' not in [n.op_type for n in fused_model.graph.node]}")

<a id='section-6'></a>
## Section 6: MatMul + Add + Relu Fusion with Memory Traffic Analysis

### Pattern

A linear layer followed by activation:

$$y = \text{ReLU}(Wx + b) = \max(0,\; Wx + b)$$

Exported as three separate nodes: `MatMul → Add → Relu`

### Memory Traffic: Unfused vs Fused

For input $x \in \mathbb{R}^{M \times K}$, weight $W \in \mathbb{R}^{K \times N}$, output $\in \mathbb{R}^{M \times N}$:

**Unfused (3 nodes):**

$$T_{\text{unfused}} = \underbrace{|x| + |W| + |h_1|}_{\text{MatMul I/O}} + \underbrace{|h_1| + |b| + |h_2|}_{\text{Add I/O}} + \underbrace{|h_2| + |y|}_{\text{Relu I/O}}$$

$$= (MK + KN + MN) + (MN + N + MN) + (MN + MN)$$

$$= MK + KN + 5MN + N$$

**Fused (1 Gemm+Relu node):**

$$T_{\text{fused}} = |x| + |W| + |b| + |y| = MK + KN + N + MN$$

**Savings:**

$$\Delta T = T_{\text{unfused}} - T_{\text{fused}} = 4MN \text{ elements} = 4MN \times \text{sizeof(dtype)}$$

For $M=32, N=1024$: savings = $4 \times 32 \times 1024 \times 4 = 512$ KB per inference!

### Fusion Data Flow

```
UNFUSED (3 kernels, 2 intermediate buffers):

  x ──┐                                         
      ├──▶ MatMul ──[write h₁]──▶ DRAM ──[read h₁]──▶ Add(b)
  W ──┘                                              │
                                         [write h₂]──┘
                                              │
                              DRAM ──[read h₂]──▶ Relu ──[write y]──▶ y

  Traffic: MK + KN + MN + MN + N + MN + MN + MN = MK + KN + 5MN + N


FUSED (1 kernel, 0 intermediate buffers):

  x ──┐
      ├──▶ FusedGemmRelu(W, b) ──[write y]──▶ y
  W ──┘     │
  b ────────┘  (bias added in-register, relu applied before write)

  Traffic: MK + KN + N + MN (all intermediates stay in registers/L1)
```

In [ ]:
# Build MatMul + Add + Relu model and demonstrate fusion
M, K, N = 32, 512, 256

W_data = np.random.randn(K, N).astype(np.float32) * 0.02
b_data = np.random.randn(N).astype(np.float32) * 0.01

W_init = numpy_helper.from_array(W_data, 'W')
b_init = numpy_helper.from_array(b_data, 'b')

matmul = helper.make_node('MatMul', ['X', 'W'], ['H1'])
add = helper.make_node('Add', ['H1', 'b'], ['H2'])
relu = helper.make_node('Relu', ['H2'], ['Y'])

X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [M, K])
Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [M, N])

graph = helper.make_graph(
    [matmul, add, relu], 'matmul_add_relu',
    inputs=[X_info], outputs=[Y_info],
    initializer=[W_init, b_init]
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
check_model(model)

print("BEFORE fusion:")
for n in model.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")

# Fuse with ORT
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
so.optimized_model_filepath = '/tmp/matmul_add_relu_fused.onnx'
sess = ort.InferenceSession(model.SerializeToString(), so)
fused_model = onnx.load('/tmp/matmul_add_relu_fused.onnx')

print(f"\nAFTER ORT fusion: {len(fused_model.graph.node)} nodes")
for n in fused_model.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")
    for attr in n.attribute:
        if attr.type == 1:  # FLOAT
            print(f"    attr {attr.name} = {attr.f}")
        elif attr.type == 2:  # INT
            print(f"    attr {attr.name} = {attr.i}")
        elif attr.type == 3:  # STRING
            print(f"    attr {attr.name} = {attr.s.decode()}")

# Memory traffic analysis
sizeof = 4  # float32
traffic_unfused = (M*K + K*N + 5*M*N + N) * sizeof
traffic_fused = (M*K + K*N + N + M*N) * sizeof
savings = traffic_unfused - traffic_fused

print(f"\n--- Memory Traffic Analysis (M={M}, K={K}, N={N}) ---")
print(f"  Unfused traffic: {traffic_unfused/1024:.1f} KB")
print(f"  Fused traffic:   {traffic_fused/1024:.1f} KB")
print(f"  Savings:         {savings/1024:.1f} KB ({100*savings/traffic_unfused:.0f}%)")

In [ ]:
# Measure actual speedup from fusion
x_input = np.random.randn(M, K).astype(np.float32)

# Session without optimization
so_none = ort.SessionOptions()
so_none.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
sess_unopt = ort.InferenceSession(model.SerializeToString(), so_none)

# Session with full optimization
so_all = ort.SessionOptions()
so_all.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_opt = ort.InferenceSession(model.SerializeToString(), so_all)

# Warmup
for _ in range(50):
    sess_unopt.run(None, {'X': x_input})
    sess_opt.run(None, {'X': x_input})

# Benchmark
n_runs = 1000

start = time.perf_counter()
for _ in range(n_runs):
    out_unopt = sess_unopt.run(None, {'X': x_input})
t_unopt = (time.perf_counter() - start) / n_runs * 1000

start = time.perf_counter()
for _ in range(n_runs):
    out_opt = sess_opt.run(None, {'X': x_input})
t_opt = (time.perf_counter() - start) / n_runs * 1000

print(f"Inference Latency (avg over {n_runs} runs):")
print(f"  Unoptimized: {t_unopt:.4f} ms")
print(f"  Optimized:   {t_opt:.4f} ms")
print(f"  Speedup:     {t_unopt/t_opt:.2f}x")

# Verify outputs match
max_diff = np.max(np.abs(out_unopt[0] - out_opt[0]))
print(f"\nNumerical equivalence: max|diff| = {max_diff:.2e}")

<a id='section-7'></a>
## Section 7: Pattern Matching and Graph Rewrite Rules

### Formalization

Graph optimizations are specified as **rewrite rules**:

$$\text{Rule}: \text{LHS} \xrightarrow{\text{condition}} \text{RHS}$$

where:
- $\text{LHS}$: a subgraph **pattern** to match
- $\text{RHS}$: the replacement subgraph
- $\text{condition}$: optional side conditions (shape constraints, attribute checks)

### Pattern Matching Algorithm

```
Algorithm: SubgraphPatternMatch(G, pattern)
─────────────────────────────────────────────
Input:  Graph G = (V, E), Pattern P = (V_p, E_p)
Output: Set of matches M ⊆ V^|V_p|

1. For each node v ∈ V:
   a. If op_type(v) == op_type(root(P)):
      b. Try to extend match from v along P's structure
      c. Check all edges, op_types, and conditions
      d. If full match found: add to M
2. Return M
```

### Example Rewrite Rules

**Rule 1: Conv+BN Fusion**
```
LHS:  X ──▶ Conv(W,b) ──▶ H ──▶ BatchNormalization(γ,β,μ,σ²) ──▶ Y
Cond: H has single consumer (the BN node)
RHS:  X ──▶ Conv(W', b') ──▶ Y
      where W' = (γ/√(σ²+ε))·W, b' = (γ/√(σ²+ε))·(b-μ) + β
```

**Rule 2: MatMul+Add → Gemm**
```
LHS:  A ──▶ MatMul(B) ──▶ H ──▶ Add(C) ──▶ Y
Cond: B is initializer, C is 1-D initializer, H has single consumer
RHS:  A ──▶ Gemm(B, C, alpha=1, beta=1) ──▶ Y
```

**Rule 3: Identity Elimination**
```
LHS:  X ──▶ Identity ──▶ Y
Cond: (none)
RHS:  Redirect all consumers of Y to use X directly
```

**Rule 4: Redundant Reshape**
```
LHS:  X ──▶ Reshape(shape₁) ──▶ H ──▶ Reshape(shape₂) ──▶ Y  
Cond: H has single consumer
RHS:  X ──▶ Reshape(shape₂) ──▶ Y
```

### ORT's Pattern Discovery Pipeline

```
┌────────────────────────────────────────────────────────────────────┐
│          ORT Graph Transformer Pipeline                            │
├────────────────────────────────────────────────────────────────────┤
│                                                                    │
│  1. Build pattern registry (compile-time)                          │
│     ├─ ConvBNFusion pattern                                        │
│     ├─ ConvReluFusion pattern                                      │
│     ├─ MatMulAddFusion pattern                                     │
│     ├─ AttentionFusion pattern (BERT/GPT)                          │
│     └─ ... (50+ registered patterns)                               │
│                                                                    │
│  2. Topological scan of graph                                      │
│     For each node v (in topo order):                               │
│       For each pattern P in registry:                              │
│         If P.match(v, graph):                                      │
│           Apply P.rewrite(v, graph)                                │
│           Break (restart scan — graph mutated)                     │
│                                                                    │
│  3. Repeat until no pattern matches (fixed point)                  │
│                                                                    │
└────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Demonstrate pattern matching: build a model with multiple fusible patterns
np.random.seed(0)

# Build a model with Conv+BN+Relu pattern (a common 3-node pattern)
C_in, C_out = 3, 16
initializers = [
    numpy_helper.from_array(np.random.randn(C_out, C_in, 3, 3).astype(np.float32)*0.1, 'W'),
    numpy_helper.from_array(np.zeros(C_out, dtype=np.float32), 'b'),
    numpy_helper.from_array(np.ones(C_out, dtype=np.float32), 'scale'),
    numpy_helper.from_array(np.zeros(C_out, dtype=np.float32), 'bias'),
    numpy_helper.from_array(np.zeros(C_out, dtype=np.float32), 'mean'),
    numpy_helper.from_array(np.ones(C_out, dtype=np.float32), 'var'),
]

nodes = [
    helper.make_node('Conv', ['X', 'W', 'b'], ['conv_out'], kernel_shape=[3,3], pads=[1,1,1,1]),
    helper.make_node('BatchNormalization', ['conv_out', 'scale', 'bias', 'mean', 'var'], ['bn_out']),
    helper.make_node('Relu', ['bn_out'], ['Y']),
]

X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, C_in, 32, 32])
Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, C_out, 32, 32])

graph = helper.make_graph(nodes, 'pattern_demo', [X_info], [Y_info], initializer=initializers)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])

print("Original graph (Conv + BN + Relu pattern):")
print(f"  Nodes: {len(model.graph.node)}")
for n in model.graph.node:
    print(f"    {n.op_type}")

# Apply pattern-based optimizations at different levels
for level_name, level in [('BASIC', ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
                           ('EXTENDED', ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
                           ('ALL', ort.GraphOptimizationLevel.ORT_ENABLE_ALL)]:
    so = ort.SessionOptions()
    so.graph_optimization_level = level
    so.optimized_model_filepath = f'/tmp/pattern_{level_name}.onnx'
    sess = ort.InferenceSession(model.SerializeToString(), so)
    opt = onnx.load(f'/tmp/pattern_{level_name}.onnx')
    print(f"\n  {level_name}: {len(opt.graph.node)} nodes")
    for n in opt.graph.node:
        attrs = {a.name: (a.s.decode() if a.type==3 else a.f if a.type==1 else a.i) 
                 for a in n.attribute}
        print(f"    {n.op_type} {attrs if attrs else ''}")

<a id='section-8'></a>
## Section 8: ORT Optimization Levels

### Level Hierarchy

| Level | Constant | Passes Included |
|:---:|:---|:---|
| 0 | `ORT_DISABLE_ALL` | No optimizations |
| 1 | `ORT_ENABLE_BASIC` | Constant folding, redundant node elimination, shape inference |
| 2 | `ORT_ENABLE_EXTENDED` | + Complex fusions (Conv+BN, Conv+Relu, Attention, GELU, etc.) |
| 99 | `ORT_ENABLE_ALL` | + Layout optimizations, all available EP-specific transforms |

### Detailed Pass Breakdown

**BASIC (Level 1) — Semantics-Preserving Simplifications:**
- `ConstantFolding`: Evaluate subgraphs with all-constant inputs
- `EliminateIdentity`: Remove no-op Identity nodes
- `EliminateSlice`: Remove slices that select the full range
- `EliminateDropout`: Remove Dropout nodes (inference mode)
- `ShapeToInitializerFolding`: Pre-compute Shape ops on known shapes
- `CommonSubexpressionElimination`: Share identical computations

**EXTENDED (Level 2) — Operator Fusions:**
- `ConvBNFusion`: Fuse Conv + BatchNormalization
- `ConvActivationFusion`: Fuse Conv + Relu/Sigmoid/Tanh
- `MatMulAddFusion`: Fuse MatMul + Add → Gemm
- `GemmActivationFusion`: Fuse Gemm + Relu
- `AttentionFusion`: Fuse multi-head attention pattern (BERT)
- `LayerNormFusion`: Fuse LayerNorm subgraph
- `GeluFusion`: Fuse GELU approximation subgraph
- `BiasGeluFusion`: Fuse Add(bias) + GELU
- `SkipLayerNormFusion`: Fuse residual + LayerNorm (transformers)

**ALL (Level 99) — Hardware-Specific:**
- Layout transformations (NCHW → NHWC for CPU/ARM)
- EP-specific kernel selection
- Memory planning and buffer reuse

### Offline vs Online Optimization

```
┌───────────────────────────────────────────────────────────────────────┐
│                                                                       │
│  OFFLINE (onnxoptimizer)           ONLINE (ORT SessionOptions)        │
│  ═══════════════════════           ═══════════════════════════         │
│                                                                       │
│  • Runs before deployment          • Runs at session creation         │
│  • Reduces .onnx file size         • EP-specific fusions              │
│  • Framework-agnostic              • Hardware-aware transforms        │
│  • Portable result                 • Tied to ORT version              │
│  • No runtime dependency           • Requires ORT runtime             │
│                                                                       │
│  Best practice: Apply BOTH                                            │
│    1. Offline: clean up exporter artifacts                            │
│    2. Online: EP-specific fusions at deployment                       │
│                                                                       │
└───────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Build a realistic multi-layer model and compare optimization levels
def build_convnet():
    """Build a small ConvNet with patterns amenable to optimization."""
    np.random.seed(0)
    initializers = []
    nodes = []

    # Layer 1: Conv + BN + Relu
    W1 = numpy_helper.from_array(np.random.randn(16, 3, 3, 3).astype(np.float32)*0.1, 'W1')
    b1 = numpy_helper.from_array(np.zeros(16, dtype=np.float32), 'b1')
    bn1_s = numpy_helper.from_array(np.ones(16, dtype=np.float32), 'bn1_s')
    bn1_b = numpy_helper.from_array(np.zeros(16, dtype=np.float32), 'bn1_b')
    bn1_m = numpy_helper.from_array(np.random.randn(16).astype(np.float32)*0.1, 'bn1_m')
    bn1_v = numpy_helper.from_array(np.ones(16, dtype=np.float32), 'bn1_v')
    initializers.extend([W1, b1, bn1_s, bn1_b, bn1_m, bn1_v])
    nodes.append(helper.make_node('Conv', ['X', 'W1', 'b1'], ['c1'], kernel_shape=[3,3], pads=[1,1,1,1]))
    nodes.append(helper.make_node('BatchNormalization', ['c1', 'bn1_s', 'bn1_b', 'bn1_m', 'bn1_v'], ['bn1']))
    nodes.append(helper.make_node('Relu', ['bn1'], ['r1']))

    # Layer 2: Conv + BN + Relu
    W2 = numpy_helper.from_array(np.random.randn(32, 16, 3, 3).astype(np.float32)*0.1, 'W2')
    b2 = numpy_helper.from_array(np.zeros(32, dtype=np.float32), 'b2')
    bn2_s = numpy_helper.from_array(np.ones(32, dtype=np.float32), 'bn2_s')
    bn2_b = numpy_helper.from_array(np.zeros(32, dtype=np.float32), 'bn2_b')
    bn2_m = numpy_helper.from_array(np.random.randn(32).astype(np.float32)*0.1, 'bn2_m')
    bn2_v = numpy_helper.from_array(np.ones(32, dtype=np.float32), 'bn2_v')
    initializers.extend([W2, b2, bn2_s, bn2_b, bn2_m, bn2_v])
    nodes.append(helper.make_node('Conv', ['r1', 'W2', 'b2'], ['c2'], kernel_shape=[3,3], pads=[1,1,1,1]))
    nodes.append(helper.make_node('BatchNormalization', ['c2', 'bn2_s', 'bn2_b', 'bn2_m', 'bn2_v'], ['bn2']))
    nodes.append(helper.make_node('Relu', ['bn2'], ['r2']))

    # Layer 3: Conv + BN + Relu
    W3 = numpy_helper.from_array(np.random.randn(64, 32, 3, 3).astype(np.float32)*0.1, 'W3')
    b3 = numpy_helper.from_array(np.zeros(64, dtype=np.float32), 'b3')
    bn3_s = numpy_helper.from_array(np.ones(64, dtype=np.float32), 'bn3_s')
    bn3_b = numpy_helper.from_array(np.zeros(64, dtype=np.float32), 'bn3_b')
    bn3_m = numpy_helper.from_array(np.random.randn(64).astype(np.float32)*0.1, 'bn3_m')
    bn3_v = numpy_helper.from_array(np.ones(64, dtype=np.float32), 'bn3_v')
    initializers.extend([W3, b3, bn3_s, bn3_b, bn3_m, bn3_v])
    nodes.append(helper.make_node('Conv', ['r2', 'W3', 'b3'], ['c3'], kernel_shape=[3,3], pads=[1,1,1,1]))
    nodes.append(helper.make_node('BatchNormalization', ['c3', 'bn3_s', 'bn3_b', 'bn3_m', 'bn3_v'], ['bn3']))
    nodes.append(helper.make_node('Relu', ['bn3'], ['r3']))

    # Global average pool + FC (MatMul + Add)
    nodes.append(helper.make_node('GlobalAveragePool', ['r3'], ['gap']))
    nodes.append(helper.make_node('Flatten', ['gap'], ['flat'], axis=1))
    W_fc = numpy_helper.from_array(np.random.randn(64, 10).astype(np.float32)*0.1, 'W_fc')
    b_fc = numpy_helper.from_array(np.zeros(10, dtype=np.float32), 'b_fc')
    initializers.extend([W_fc, b_fc])
    nodes.append(helper.make_node('MatMul', ['flat', 'W_fc'], ['fc_out']))
    nodes.append(helper.make_node('Add', ['fc_out', 'b_fc'], ['Y']))

    X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 3, 32, 32])
    Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 10])
    graph = helper.make_graph(nodes, 'convnet', [X_info], [Y_info], initializer=initializers)
    return helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])

model = build_convnet()
print(f"Original model: {len(model.graph.node)} nodes")
print("Op types:", [n.op_type for n in model.graph.node])

# Compare all optimization levels
results = {}
levels = [
    ('DISABLED', ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
    ('BASIC', ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
    ('EXTENDED', ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
    ('ALL', ort.GraphOptimizationLevel.ORT_ENABLE_ALL),
]

print("\n" + "="*60)
for name, level in levels:
    so = ort.SessionOptions()
    so.graph_optimization_level = level
    outpath = f'/tmp/opt_{name}.onnx'
    so.optimized_model_filepath = outpath
    sess = ort.InferenceSession(model.SerializeToString(), so)
    opt = onnx.load(outpath)
    results[name] = len(opt.graph.node)
    op_types = [n.op_type for n in opt.graph.node]
    print(f"\n{name} (level={level.numerator}): {len(opt.graph.node)} nodes")
    for op in sorted(set(op_types)):
        print(f"    {op}: {op_types.count(op)}")

In [ ]:
# Visualize optimization effectiveness across levels
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Node count by optimization level
level_names = list(results.keys())
node_counts = list(results.values())
colors = ['#FF6B6B', '#FFA500', '#4ECDC4', '#45B7D1']

bars = axes[0].bar(level_names, node_counts, color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_ylabel('Number of Nodes', fontsize=12)
axes[0].set_xlabel('ORT Optimization Level', fontsize=12)
axes[0].set_title('Graph Size vs Optimization Level', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for bar, count in zip(bars, node_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)

# Plot 2: Typical pass effectiveness
passes = ['Constant\nFolding', 'Dead Code\nElim.', 'Identity\nRemoval',
          'Conv+BN\nFusion', 'Conv+Relu\nFusion', 'MatMul+Add\n→Gemm',
          'LayerNorm\nFusion']
typical_reduction = [15, 8, 5, 20, 12, 10, 25]

bars2 = axes[1].barh(passes, typical_reduction, color='#6C5CE7', edgecolor='black', linewidth=1.2)
axes[1].set_xlabel('Typical Node Reduction (%)', fontsize=12)
axes[1].set_title('Optimization Pass Effectiveness\n(Typical Models)', fontsize=13, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
for bar, val in zip(bars2, typical_reduction):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('optimization_levels.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# List available onnxoptimizer passes and categorize them
passes = onnxoptimizer.get_available_passes()
print(f"Available onnxoptimizer passes ({len(passes)} total):")
print("=" * 60)

categories = {
    'Elimination': [p for p in passes if 'eliminate' in p],
    'Fusion': [p for p in passes if 'fuse' in p],
    'Extraction': [p for p in passes if 'extract' in p],
    'Lifting': [p for p in passes if 'lift' in p],
    'Other': [p for p in passes if not any(k in p for k in ['eliminate', 'fuse', 'extract', 'lift'])],
}

for cat, cat_passes in categories.items():
    if cat_passes:
        print(f"\n{cat} ({len(cat_passes)} passes):")
        for p in sorted(cat_passes):
            print(f"  • {p}")

<a id='section-9'></a>
## Section 9: Correctness Proofs and Semantic Equivalence

### The Semantic Equivalence Requirement

Every graph optimization must satisfy:

$$\forall \mathbf{x} \in \mathcal{X}: \|f_{G}(\mathbf{x}) - f_{G'}(\mathbf{x})\|_\infty \leq \epsilon_{\text{fp}}$$

where $\epsilon_{\text{fp}}$ is the floating-point tolerance bound.

### Theorem: Fusion Correctness

**Claim:** If $f = g \circ h$ (composition of two functions), and we have a fused implementation $f_{\text{fused}}$ that computes the same mathematical function, then:

$$\|f_{\text{fused}}(\mathbf{x}) - f_{\text{original}}(\mathbf{x})\| \leq \epsilon_{\text{fp}}$$

**Proof for Conv+BN fusion:**

Let $\hat{y} = \text{Conv}_{W',b'}(x)$ and $y = \text{BN}(\text{Conv}_{W,b}(x))$.

From our derivation, $\hat{y}$ and $y$ compute the **same mathematical expression**:

$$\hat{y}_c = \sum_{c'} W'_{c,c'} * x_{c'} + b'_c = \sum_{c'} s_c W_{c,c'} * x_{c'} + s_c(b_c - \mu_c) + \beta_c$$

The only source of discrepancy is **floating-point evaluation order**:
- Original: multiply, accumulate, subtract $\mu$, divide by $\sqrt{\sigma^2+\epsilon}$, multiply by $\gamma$, add $\beta$
- Fused: multiply by pre-scaled weights, accumulate, add pre-computed bias

By IEEE 754 error analysis, for an expression with $n$ floating-point operations:

$$|\hat{y} - y| \leq n \cdot \mathbf{u} \cdot |y| + O(\mathbf{u}^2)$$

where $\mathbf{u} = 2^{-24} \approx 5.96 \times 10^{-8}$ is the unit roundoff for float32.

### Bound for Conv+BN Fusion

The fused computation has fewer operations than the original (no separate BN pass), so:

$$\|\hat{y} - y\|_\infty \leq C_{\text{in}} \cdot k_h \cdot k_w \cdot \mathbf{u} \cdot \|y\|_\infty$$

For typical CNN layers ($C_{\text{in}}=64, k=3$): bound $\approx 576 \cdot 6 \times 10^{-8} \approx 3.5 \times 10^{-5}$.

### Practical Tolerance Guidelines

| Dtype | Recommended $\epsilon_{\text{tol}}$ | Justification |
|:---:|:---:|:---|
| float32 | $10^{-5}$ | ~3 ULPs for typical layer sizes |
| float16 | $10^{-2}$ | Half-precision unit roundoff is $2^{-11}$ |
| int8 (quantized) | $1$ | Integer quantization inherently lossy |

In [ ]:
# Full verification protocol: original vs optimized model
def verify_equivalence(original_model, optimized_model, input_shapes,
                       n_samples=200, tol=1e-5):
    """Verify semantic equivalence between original and optimized models."""
    sess_orig = ort.InferenceSession(original_model.SerializeToString(),
                                     providers=['CPUExecutionProvider'])
    sess_opt = ort.InferenceSession(optimized_model.SerializeToString(),
                                    providers=['CPUExecutionProvider'])

    input_names = [inp.name for inp in sess_orig.get_inputs()]
    max_diffs = []
    mean_diffs = []

    for i in range(n_samples):
        feeds = {}
        for name, shape in zip(input_names, input_shapes):
            feeds[name] = np.random.randn(*shape).astype(np.float32)

        out_orig = sess_orig.run(None, feeds)
        out_opt = sess_opt.run(None, feeds)

        max_diff = max(np.max(np.abs(o1 - o2)) for o1, o2 in zip(out_orig, out_opt))
        mean_diff = max(np.mean(np.abs(o1 - o2)) for o1, o2 in zip(out_orig, out_opt))
        max_diffs.append(max_diff)
        mean_diffs.append(mean_diff)

    return np.array(max_diffs), np.array(mean_diffs)

# Build and optimize
original = build_convnet()
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.optimized_model_filepath = '/tmp/verified_opt.onnx'
_ = ort.InferenceSession(original.SerializeToString(), so)
optimized = onnx.load('/tmp/verified_opt.onnx')

# Run verification
max_diffs, mean_diffs = verify_equivalence(original, optimized, [(1, 3, 32, 32)])

print("╔══════════════════════════════════════════════════════╗")
print("║       Numerical Equivalence Verification Report      ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Samples tested:        {len(max_diffs):>6}                      ║")
print(f"║  Original nodes:        {len(original.graph.node):>6}                      ║")
print(f"║  Optimized nodes:       {len(optimized.graph.node):>6}                      ║")
print(f"║  Node reduction:        {len(original.graph.node)-len(optimized.graph.node):>6} ({100*(1-len(optimized.graph.node)/len(original.graph.node)):.0f}%)                 ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Max |diff|  (worst):   {max_diffs.max():.2e}                  ║")
print(f"║  Max |diff|  (mean):    {max_diffs.mean():.2e}                  ║")
print(f"║  Mean |diff| (mean):    {mean_diffs.mean():.2e}                  ║")
print(f"║  Tolerance:             1.00e-05                    ║")
print(f"║  ALL within tolerance:  {str(np.all(max_diffs < 1e-5)):>5}                     ║")
print("╚══════════════════════════════════════════════════════╝")

In [ ]:
# Visualize the distribution of numerical differences
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of max absolute differences
axes[0].hist(max_diffs, bins=30, color='#6C5CE7', edgecolor='black', alpha=0.85)
axes[0].axvline(1e-5, color='red', linestyle='--', linewidth=2, label=r'Tolerance $\epsilon_{tol}=10^{-5}$')
axes[0].set_xlabel('Max Absolute Difference per Sample', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Distribution of Max |Original - Optimized|', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Before/after comparison + speedup
categories = ['Original\n(unoptimized)', 'Optimized\n(ORT_ENABLE_ALL)']
counts = [len(original.graph.node), len(optimized.graph.node)]
bars = axes[1].bar(categories, counts, color=['#FF6B6B', '#4ECDC4'],
                   edgecolor='black', linewidth=1.5, width=0.5)
axes[1].set_ylabel('Node Count', fontsize=11)
axes[1].set_title('Graph Complexity: Before vs After', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

for bar, count in zip(bars, counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                str(count), ha='center', fontweight='bold', fontsize=14)

reduction = (1 - counts[1]/counts[0]) * 100
axes[1].annotate(f'{reduction:.0f}% fewer nodes\n(semantically equivalent)',
                xy=(0.5, (counts[0]+counts[1])/2),
                fontsize=11, ha='center', color='darkgreen', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('verification_results.png', dpi=100, bbox_inches='tight')
plt.show()

<a id='section-10'></a>
## Section 10: Practical Implementation

### End-to-End Optimization Workflow

```
┌────────────────────────────────────────────────────────────────────┐
│           Production Optimization Workflow                          │
├────────────────────────────────────────────────────────────────────┤
│                                                                    │
│  1. Export model to ONNX (torch.onnx.export / tf2onnx)            │
│  2. Validate: onnx.checker.check_model(model)                     │
│  3. Offline optimization: onnxoptimizer.optimize(model, passes)   │
│  4. Save optimized model                                           │
│  5. Online optimization: ORT SessionOptions at deployment          │
│  6. Verify: compare outputs on representative inputs               │
│  7. Benchmark: measure latency improvement                         │
│                                                                    │
└────────────────────────────────────────────────────────────────────┘
```

### Key APIs

```python
# Offline (onnxoptimizer)
import onnxoptimizer
optimized = onnxoptimizer.optimize(model, passes=['eliminate_deadend',
    'eliminate_identity', 'fuse_bn_into_conv', 'fuse_matmul_add_bias_into_gemm'])

# Online (ORT)
import onnxruntime as ort
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.optimized_model_filepath = 'optimized.onnx'  # Save for inspection
session = ort.InferenceSession('model.onnx', so)
```

In [ ]:
# Complete practical workflow: optimize, save, verify, benchmark

# Step 1: Build a model (simulating an export)
model = build_convnet()
onnx.save(model, '/tmp/original_model.onnx')
print("Step 1: Original model")
print(f"  Nodes: {len(model.graph.node)}")
print(f"  File size: {len(model.SerializeToString())/1024:.1f} KB")

# Step 2: Offline optimization with onnxoptimizer
offline_passes = [
    'eliminate_deadend',
    'eliminate_identity',
    'eliminate_nop_dropout',
    'eliminate_nop_transpose',
    'fuse_bn_into_conv',
    'fuse_consecutive_transposes',
    'fuse_matmul_add_bias_into_gemm',
]

# Filter to available passes
available = set(onnxoptimizer.get_available_passes())
valid_passes = [p for p in offline_passes if p in available]
print(f"\nStep 2: Offline optimization ({len(valid_passes)} passes)")

offline_opt = onnxoptimizer.optimize(model, valid_passes)
check_model(offline_opt)
onnx.save(offline_opt, '/tmp/offline_optimized.onnx')
print(f"  Nodes after offline: {len(offline_opt.graph.node)}")
print(f"  File size: {len(offline_opt.SerializeToString())/1024:.1f} KB")

# Step 3: Online optimization with ORT
print(f"\nStep 3: Online optimization (ORT_ENABLE_ALL)")
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.optimized_model_filepath = '/tmp/fully_optimized.onnx'
sess = ort.InferenceSession(offline_opt.SerializeToString(), so)
final_opt = onnx.load('/tmp/fully_optimized.onnx')
print(f"  Nodes after ORT: {len(final_opt.graph.node)}")
print(f"  File size: {len(final_opt.SerializeToString())/1024:.1f} KB")

# Step 4: Summary
print(f"\n{'='*50}")
print(f"OPTIMIZATION SUMMARY")
print(f"{'='*50}")
print(f"  Original:        {len(model.graph.node):>3} nodes")
print(f"  After offline:   {len(offline_opt.graph.node):>3} nodes (-{len(model.graph.node)-len(offline_opt.graph.node)})")
print(f"  After ORT:       {len(final_opt.graph.node):>3} nodes (-{len(offline_opt.graph.node)-len(final_opt.graph.node)})")
print(f"  Total reduction: {len(model.graph.node)-len(final_opt.graph.node)} nodes ({100*(1-len(final_opt.graph.node)/len(model.graph.node)):.0f}%)")

In [ ]:
# Benchmark: measure latency improvement from optimization
x_input = np.random.randn(1, 3, 32, 32).astype(np.float32)

# Create sessions at different optimization levels
sessions = {}
for name, level in [('DISABLED', ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
                     ('BASIC', ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
                     ('EXTENDED', ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
                     ('ALL', ort.GraphOptimizationLevel.ORT_ENABLE_ALL)]:
    so = ort.SessionOptions()
    so.graph_optimization_level = level
    sessions[name] = ort.InferenceSession(model.SerializeToString(), so,
                                          providers=['CPUExecutionProvider'])

# Warmup all sessions
for sess in sessions.values():
    for _ in range(100):
        sess.run(None, {'X': x_input})

# Benchmark
n_runs = 2000
latencies = {}

for name, sess in sessions.items():
    start = time.perf_counter()
    for _ in range(n_runs):
        sess.run(None, {'X': x_input})
    elapsed = (time.perf_counter() - start) / n_runs * 1000
    latencies[name] = elapsed

print(f"Inference Latency (avg over {n_runs} runs, batch=1, 32x32 input):")
print(f"{'='*50}")
baseline = latencies['DISABLED']
for name, lat in latencies.items():
    speedup = baseline / lat
    print(f"  {name:>10}: {lat:.4f} ms  (speedup: {speedup:.2f}x)")

In [ ]:
# Visualize latency comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Latency by optimization level
names = list(latencies.keys())
times = list(latencies.values())
colors = ['#FF6B6B', '#FFA500', '#4ECDC4', '#45B7D1']

bars = axes[0].bar(names, times, color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_ylabel('Latency (ms)', fontsize=12)
axes[0].set_xlabel('Optimization Level', fontsize=12)
axes[0].set_title('Inference Latency by Optimization Level', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for bar, t in zip(bars, times):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{t:.3f}ms', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 2: Speedup relative to DISABLED
speedups = [baseline/t for t in times]
bars2 = axes[1].bar(names, speedups, color=colors, edgecolor='black', linewidth=1.2)
axes[1].set_ylabel('Speedup (×)', fontsize=12)
axes[1].set_xlabel('Optimization Level', fontsize=12)
axes[1].set_title('Speedup Relative to No Optimization', fontsize=13, fontweight='bold')
axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(axis='y', alpha=0.3)
for bar, s in zip(bars2, speedups):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{s:.2f}×', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('latency_benchmark.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Inspect the optimized model: what nodes remain and what was fused
print("Optimized Model Inspection (ORT_ENABLE_ALL)")
print("=" * 55)

final_opt = onnx.load('/tmp/fully_optimized.onnx')

print(f"\nGraph: {final_opt.graph.name}")
print(f"Nodes: {len(final_opt.graph.node)}")
print(f"Initializers: {len(final_opt.graph.initializer)}")
print(f"\nNode details:")
for i, n in enumerate(final_opt.graph.node):
    print(f"  [{i}] {n.op_type} (domain: '{n.domain or 'ai.onnx'}')")
    print(f"      inputs:  {list(n.input)}")
    print(f"      outputs: {list(n.output)}")
    if n.attribute:
        for attr in n.attribute:
            if attr.type == 1:
                print(f"      {attr.name} = {attr.f}")
            elif attr.type == 2:
                print(f"      {attr.name} = {attr.i}")
            elif attr.type == 3:
                print(f"      {attr.name} = '{attr.s.decode()}'")
            elif attr.type == 7:
                print(f"      {attr.name} = {list(attr.ints)}")

# Check which original ops were eliminated
original_ops = set(n.op_type for n in model.graph.node)
optimized_ops = set(n.op_type for n in final_opt.graph.node)
eliminated = original_ops - optimized_ops
new_ops = optimized_ops - original_ops

print(f"\nOps eliminated: {eliminated or 'none'}")
print(f"New fused ops:  {new_ops or 'none'}")

In [ ]:
# Applying individual onnxoptimizer passes to see incremental effects
model = build_convnet()

individual_passes = [
    'eliminate_identity',
    'eliminate_deadend',
    'eliminate_nop_dropout',
    'fuse_bn_into_conv',
    'fuse_matmul_add_bias_into_gemm',
    'fuse_consecutive_transposes',
]

available_passes = set(onnxoptimizer.get_available_passes())

print("Incremental pass application:")
print(f"{'Pass':<40} {'Nodes':>6} {'Δ':>4}")
print("-" * 55)
print(f"{'(original)':<40} {len(model.graph.node):>6} {'':>4}")

current = model
for pass_name in individual_passes:
    if pass_name not in available_passes:
        print(f"{pass_name:<40} {'(unavailable)':>6}")
        continue
    prev_count = len(current.graph.node)
    try:
        current = onnxoptimizer.optimize(current, [pass_name])
        delta = len(current.graph.node) - prev_count
        sign = '+' if delta > 0 else '' if delta == 0 else ''
        print(f"{pass_name:<40} {len(current.graph.node):>6} {sign}{delta:>3}")
    except Exception as e:
        print(f"{pass_name:<40} {'error':>6} {str(e)[:20]}")

print("-" * 55)
print(f"{'Total reduction':<40} {len(model.graph.node) - len(current.graph.node):>6} nodes")

<a id='section-11'></a>
## Summary

### Key Optimizations Covered

| Optimization | Formal Basis | Typical Benefit |
|:---|:---|:---|
| Constant Folding | Partial evaluation: $\text{PE}(G, \sigma) \to G'$ | Fewer runtime ops |
| Dead Code Elimination | Backward reachability on DAG | Smaller graph |
| Identity Removal | Tensor aliasing | Reduced overhead |
| Conv+BN Fusion | $W' = \frac{\gamma}{\sqrt{\sigma^2+\epsilon}} W$ | Eliminate BN layer entirely |
| MatMul+Add+Relu | Gemm+activation kernel | $4MN$ bytes saved |
| LayerNorm Fusion | 8 nodes → 1 | Major transformer speedup |

### Critical Formulas

**Conv+BN Fusion:**

$$W' = \frac{\gamma}{\sqrt{\sigma^2+\epsilon}} \cdot W, \quad b' = \frac{\gamma(b-\mu)}{\sqrt{\sigma^2+\epsilon}} + \beta$$

**Memory Savings (k-op fusion):**

$$\text{Savings} = (k-1) \times 2 \times |T| \times \text{sizeof(dtype)}$$

**Correctness Guarantee:**

$$\forall \mathbf{x}: \|f_{\text{fused}}(\mathbf{x}) - f_{\text{original}}(\mathbf{x})\|_\infty \leq \epsilon_{\text{fp}}$$

### Interview Takeaways

1. **Constant folding** = partial evaluation; provably correct by determinism + immutability
2. **Conv+BN fusion** = algebraic absorption; know the $W', b'$ formulas
3. **Fusion benefit** = eliminated memory traffic for intermediates (not compute)
4. **ORT levels**: BASIC (folding, elimination), EXTENDED (fusions), ALL (layout)
5. **Always verify** numerical equivalence after optimization

---

**Next:** [Graph Optimizations — Apply](./Graph_Optimizations_Apply.ipynb) | [Quantization Techniques](../02_Quantization_Techniques/)